# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [ ]:
import os, subprocess, sys

# ── 1. Install uv ────────────────────────────────────────────────────────────
subprocess.run("wget -qO- https://astral.sh/uv/install.sh | sh", shell=True)
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")

# ── 2. Create venv ──────────────────────────────────────────────────────────
subprocess.run("uv venv .venv --seed --clear --python 3.11", shell=True)
pip = ".venv/bin/python -m pip"

# ── 3. Phase 1: Blackwell PyTorch (The Critical Fix) ─────────────────────────
# We use CUDA 12.8 nightly because it contains the sm_120 kernels.
print("Installing Blackwell-ready PyTorch (CUDA 12.8 nightly)...")
subprocess.run(f"""
{pip} install --pre torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/nightly/cu128 --force-reinstall
""", shell=True, check=True)

# ── 4. Phase 2: Math Reasoning Stack ─────────────────────────────────────────
# We install these without version pins for the most part to allow the 
# resolver to find Blackwell-compatible wheels (Qwen-2507 needs newer libs).
print("Installing remaining math-reasoning stack...")
subprocess.run(f"""
{pip} install \
    "numpy<2" \
    "transformers>=4.51.0" \
    "accelerate>=0.30" \
    "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "vllm>=0.7.0" \
    "triton" \
    "nvidia-cuda-runtime-cu12" \
    "nvidia-cuda-cupti-cu12" \
    "nvidia-cudnn-cu12" \
    "nvidia-nvjitlink-cu12" \
    "pandas" \
    "antlr4-python3-runtime==4.11.1" \
    sympy tqdm sentencepiece ipykernel jupyter
""", shell=True, check=True)

# ── 5. Environment & Kernel Setup ───────────────────────────────────────────
venv_site = ".venv/lib/python3.11/site-packages"
ld_paths = [
    f"{venv_site}/nvidia/nvjitlink/lib",
    f"{venv_site}/nvidia/cuda_runtime/lib",
    f"{venv_site}/nvidia/cudnn/lib",
    "/usr/local/cuda/lib64",
    "/usr/local/cuda-12.8/lib64",
]
os.environ["LD_LIBRARY_PATH"] = ":".join(ld_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Permanently fix bashrc for background tasks and future sessions
bashrc_line = f'\nexport LD_LIBRARY_PATH={":".join(ld_paths)}:$LD_LIBRARY_PATH\n'
with open(os.path.expanduser("~/.bashrc"), "a") as f:
    f.write(bashrc_line)

subprocess.run(
    ".venv/bin/python -m ipykernel install --user --name cse151bV2 --display-name 'Python (cse151bV2)'",
    shell=True
)
print("\n✅ Environment built for Blackwell (sm_120).")
print("👉 ACTION: Select kernel 'cse151bV2' in the top right → RESTART KERNEL.")

In [1]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "data/public_with_reasoning.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 8000

DEVICE = "cuda"

In [2]:
import gc, torch

try: del llm
except: pass
try: del train_model
except: pass
try: del base
except: pass
try: del model_merged
except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

GPU free: 25.1 GB


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
#load data
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

#load model 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
)

train_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Thinking-2507",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

train_model = prepare_model_for_kbit_training(train_model)

#Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.06,
    bias="none",
    task_type="CAUSAL_LM"
)

train_model = get_peft_model(train_model, lora_config)

train_model.print_trainable_parameters()

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
    padding_side="left",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", tokenizer.__class__.__name__)

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning. Think concisely. Step by step.\n\n "
    "MANDATORY RULE:\n"
    "- Show ALL reasoning inside <think>...</think> tags."
    "- The FINAL token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the final token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{final_answer}\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician reasoning. Think concisely. Step by step.\n\n"
    "MANDATORY RULE:\n"
    "- Show ALL reasoning inside <think>...</think> tags."
    "- The FINAL token MUST be: \\boxed{X} where X is the answer choice given the options A-Z.\n "
    "- If this is not the final token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{X}\n"
)

SYSTEM_PROMPT_REFLECTION_FREE = (
    "The previous response did NOT provide a valid boxed answer. You MUST provide one now.\n\n"
    "MANDATORY RULE:\n"
    "- Re-examine the problem from scratch.\n"
    "- Show reasoning inside <think>...</think> tags.\n"
    "- You MUST end with \\boxed{answer} - this is REQUIRED.\n"
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n"
    "- FAILURE TO PROVIDE \\boxed{} IS NOT ACCEPTABLE.\n"
)

SYSTEM_PROMPT_REFLECTION_MCQ = (
    "The previous response did NOT provide a valid boxed answer. You MUST provide one now.\n\n"
    "MANDATORY RULE:\n"
    "- Re-examine the problem from scratch.\n"
    "- Show reasoning inside <think>...</think> tags.\n"
    "- You MUST end with \\boxed{X} where X is the answer choice (A-Z) - this is REQUIRED.\n"
    "- FAILURE TO PROVIDE \\boxed{} IS NOT ACCEPTABLE.\n"
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def build_reflection_prompt(question: str, options: Optional[list]) -> tuple[str, str, str]:
    """Return (system_prompt, user_prompt, reflection_prompt) for a question."""
    system_prompt, user_prompt = build_prompt(question, options)
    if options:
        return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_MCQ
    return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_FREE

In [ ]:
import os, sys, subprocess

os.environ["VLLM_ATTENTION_BACKEND"] = "TRITON_ATTN"

cmd = [
    sys.executable, "generate_reasoning.py",
    "--data-path", "data/public.jsonl",
    "--output-path", "data/public_with_reasoning.jsonl",
    "--model-id", "Qwen/Qwen3-4B-Thinking-2507",
    "--k", "8",
    "--max-model-len", "8192",
    "--max-new-tokens", "3000",
]

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                           env={**os.environ})
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()

In [ ]:
from torch.utils.data import Dataset
import re

class MathDataset(Dataset):
    """
    Tokenizes question->answer pairs for supervised fine-tuning.

    Training target per sample:
        <think>
        {real reasoning from item["reasoning"]}
        </think>

        \\boxed{answer}

    Reasoning is trimmed to max_think_chars at a sentence boundary so
    the model learns concise CoT rather than rambling.
    """

    MAX_THINK_CHARS = 600  # tune: shorter = punchier reasoning

    def __init__(self, data, tokenizer, max_length=1024):
        self.samples = []
        skipped = 0

        for item in data:
            system, user = build_prompt(item["question"], item.get("options"))

            answer = item["answer"]
            if isinstance(answer, list):
                answer = ", ".join(str(a) for a in answer)

            # ── Reasoning: use real CoT if available, else short fallback ──
            reasoning = item.get("think", "Let me solve this step by step.")
            # if len(reasoning) > self.MAX_THINK_CHARS:
            #     # Trim at sentence boundary to avoid cutting mid-word
            #     trimmed = reasoning[: self.MAX_THINK_CHARS].rsplit(".", 1)[0]
            #     reasoning = trimmed + "." if trimmed else reasoning[: self.MAX_THINK_CHARS]

            assistant_content = (
                f"<think>\n{reasoning}\n</think>\n\n"
                f"\\boxed{{{answer}}}"
            )

            messages = [
                {"role": "system",    "content": system},
                {"role": "user",      "content": user},
                {"role": "assistant", "content": assistant_content},
            ]

            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            tokenized = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding=False,
                return_tensors="pt",
            )

            input_ids      = tokenized["input_ids"].squeeze()
            attention_mask = tokenized["attention_mask"].squeeze()  # needed by trainer
            labels         = input_ids.clone()

            # Mask the prompt (system + user) — only train on assistant output
            assistant_header_tokens = tokenizer.encode(
                "<|im_start|>assistant\n", add_special_tokens=False
            )
            masked = False
            for i in range(len(labels) - len(assistant_header_tokens)):
                if labels[i : i + len(assistant_header_tokens)].tolist() == assistant_header_tokens:
                    labels[: i + len(assistant_header_tokens)] = -100
                    masked = True
                    break

            if not masked:
                # Header not found after tokenization — skip rather than train on garbage
                skipped += 1
                continue

            self.samples.append({
                "input_ids":      input_ids,
                "attention_mask": attention_mask,
                "labels":         labels,
            })

        print(f"MathDataset: {len(self.samples)} samples built, {skipped} skipped (no assistant header found)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


train_dataset = MathDataset(data, tokenizer, max_length=2048)
print(f"Training samples: {len(train_dataset)}")


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="data/lora_math_adapter",
    num_train_epochs=4,
    per_device_train_batch_size=2,     
    gradient_accumulation_steps=8,     
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=train_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=train_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
ADAPTER_PATH = "data/lora_math_adapter/final_adapter"
train_model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}")

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
#restart before continuing
#rerun everything that doesnt result in retraining

In [3]:
import gc

# Release train_model (LoRA-wrapped model used during fine-tuning)
try:
    train_model.cpu()
    del train_model
    print("train_model deleted")
except NameError:
    pass

# Release HF model if it was loaded separately
try:
    model.cpu()
    del model
    print("model deleted")
except NameError:
    pass

# Release tokenizer
try:
    del tokenizer
    print("tokenizer deleted")
except NameError:
    pass

gc.collect()

import torch
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory freed. Allocated: 0.00 GB


In [4]:
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
ADAPTER_PATH = "data/lora_math_adapter/final_adapter"
MERGED_PATH = "/tmp/lora_math_merged"

In [5]:
USE_PRIVATE = True
DATA_PATH = "data/private.jsonl" if USE_PRIVATE else "data/public_with_reasoning.jsonl"
data = [json.loads(line) for line in open(DATA_PATH)]
print(f"Loaded {len(data)} examples from {'PRIVATE' if USE_PRIVATE else 'PUBLIC'} set")

Loaded 943 examples from PRIVATE set


In [6]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

GPU free: 25.1 GB


In [ ]:
#if base not found in cache
from transformers import AutoModelForCausalLM, AutoTokenizer
model_id = "Qwen/Qwen3-4B-Thinking-2507" 
AutoTokenizer.from_pretrained(model_id)
AutoModelForCausalLM.from_pretrained(model_id)

In [7]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MERGED_PATH = "/tmp/lora_math_merged"

print("1. Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="cuda",
    local_files_only=True,
)
print("2. Loading adapter...")
model_merged = PeftModel.from_pretrained(base, ADAPTER_PATH)
print("3. Merging...")
model_merged = model_merged.merge_and_unload()
print("4. Saving...")
model_merged.save_pretrained(MERGED_PATH)
AutoTokenizer.from_pretrained(ADAPTER_PATH).save_pretrained(MERGED_PATH)
print("Done.")

`torch_dtype` is deprecated! Use `dtype` instead!


1. Loading base model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

2. Loading adapter...


3. Merging...
4. Saving...


Done.


In [8]:
import gc, torch

try: del base
except: pass
try: del model_merged
except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

Free GPU: 25.1 GB


In [9]:
import torch
print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

Allocated: 0.01 GB
Reserved:  0.02 GB


In [10]:
from vllm import LLM, SamplingParams
MERGED_PATH = "/tmp/lora_math_merged"

llm = LLM(
    model=MERGED_PATH,
    dtype="half",
    gpu_memory_utilization=0.85,
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=32,
)
lora_request = None

INFO 05-20 05:08:03 [utils.py:263] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 32, 'disable_log_stats': True, 'model': '/tmp/lora_math_merged'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 05-20 05:08:03 [model.py:530] Resolved architecture: Qwen3ForCausalLM


INFO 05-20 05:08:03 [model.py:1545] Using max model len 8192


...

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 11/11 [00:00<00:00, 22.62it/s]
Capturing CUDA graphs (decode, FULL):  14%|█▍        | 1/7 [00:00<00:01,  5.02it/s]

Capturing CUDA graphs (decode, FULL): 100%|██████████| 7/7 [00:00<00:00, 16.29it/s]


(EngineCore_DP0 pid=646) INFO 05-20 05:08:45 [gpu_model_runner.py:4856] Graph capturing finished in 2 secs, took 0.10 GiB
(EngineCore_DP0 pid=646) INFO 05-20 05:08:45 [core.py:273] init engine (profile, create kv cache, warmup model) took 20.84 seconds


(EngineCore_DP0 pid=646) The tokenizer you are loading from '/tmp/lora_math_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore_DP0 pid=646) INFO 05-20 05:08:46 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 05-20 05:08:46 [llm.py:347] Supported tasks: ['generate']


In [11]:
# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )
from vllm import SamplingParams
sampling_params_initial = SamplingParams(
    max_tokens=8000,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
)

sampling_params_step2 = SamplingParams(
    max_tokens=3000,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
)

In [12]:
def generate_batch(prompts, sampling_params):
    outputs = llm.generate(prompts, sampling_params)
    return [o.outputs[0].text.strip() for o in outputs]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [13]:
# def extract_boxed(text):
#     m = re.search(r"\\boxed\{.*?\}", text)
#     if m:
#         return m.group(0)
#     return text[:200]


def extract_boxed(text):
    match = re.search(r"\\boxed\{", text)
    if not match:
        return text[:200]
    start = match.end()
    depth, i = 1, start
    while i < len(text) and depth:
        if text[i] == '{': depth += 1
        elif text[i] == '}': depth -= 1
        i += 1
    return text[match.start():i] if depth == 0 else text[:200]

In [14]:
from transformers import AutoTokenizer
MERGED_PATH = "/tmp/lora_math_merged"
tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH, trust_remote_code=True)
temp_output_file = "data/responses_checkpoint.csv"

The tokenizer you are loading from '/tmp/lora_math_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [15]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning. Think concisely. Step by step.\n\n "
    "MANDATORY RULE:\n"
    "- Show ALL reasoning inside <think>...</think> tags."
    "- The FINAL token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the final token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{final_answer}\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician reasoning. Think concisely. Step by step.\n\n"
    "MANDATORY RULE:\n"
    "- Show ALL reasoning inside <think>...</think> tags."
    "- The FINAL token MUST be: \\boxed{X} where X is the answer choice given the options A-Z.\n "
    "- If this is not the final token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{X}\n"
)

SYSTEM_PROMPT_REFLECTION_FREE = (
    "The previous response did NOT provide a valid boxed answer. You MUST provide one now.\n\n"
    "MANDATORY RULE:\n"
    "- Re-examine the problem from scratch.\n"
    "- Show reasoning inside <think>...</think> tags.\n"
    "- You MUST end with \\boxed{answer} - this is REQUIRED.\n"
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n"
    "- FAILURE TO PROVIDE \\boxed{} IS NOT ACCEPTABLE.\n"
)

SYSTEM_PROMPT_REFLECTION_MCQ = (
    "The previous response did NOT provide a valid boxed answer. You MUST provide one now.\n\n"
    "MANDATORY RULE:\n"
    "- Re-examine the problem from scratch.\n"
    "- Show reasoning inside <think>...</think> tags.\n"
    "- You MUST end with \\boxed{X} where X is the answer choice (A-Z) - this is REQUIRED.\n"
    "- FAILURE TO PROVIDE \\boxed{} IS NOT ACCEPTABLE.\n"
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def build_reflection_prompt(question: str, options: Optional[list]) -> tuple[str, str, str]:
    """Return (system_prompt, user_prompt, reflection_prompt) for a question."""
    system_prompt, user_prompt = build_prompt(question, options)
    if options:
        return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_MCQ
    return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_FREE

In [16]:
import pandas as pd
import re

# ── Helpers ───────────────────────────────────────────────────────────────────

def has_boxed_in_tail(text: str, tail_words: int = 150) -> bool:
    words = text.split()
    tail  = " ".join(words[-tail_words:])
    return bool(re.search(r"\\boxed\{.+?\}", tail))


def reflection_answer_is_novel(reflection_text: str, question: str, options) -> bool:
    m = re.search(r"\\boxed\{(.+?)\}", reflection_text)
    if not m:
        return False
    boxed_val = m.group(1).strip()
    if boxed_val.lower() in ("final_answer", "x", ""):
        return False
    if not options:
        return True
    valid_letters = {chr(65 + i) for i in range(len(options))}
    return boxed_val.upper() in valid_letters


SYSTEM_PROMPT_CONTINUE = (
    "You did not finish. Continue your reasoning from exactly where you left off "
    "and end with \\boxed{answer}. Do NOT restart or repeat work already done."
)

SYSTEM_PROMPT_VERIFY = (
    "You provided an answer. Now double-check your work step by step.\n"
    "- If you find an error, correct it and end with the new \\boxed{answer}.\n"
    "- If your answer is correct, restate \\boxed{answer}.\n"
    "Either way you MUST end with \\boxed{answer}."
)


# ── Checkpoint setup ─────────────────────────────────────────────────────────

if os.path.exists(temp_output_file):
    df_existing   = pd.read_csv(temp_output_file)
    completed_ids = set(df_existing["id"].tolist())
    print(f"Resuming from checkpoint. {len(completed_ids)} items already processed.")
else:
    completed_ids = set()
    pd.DataFrame(
        columns=["id", "question", "initial_response", "step2_response", "final_response", "has_box"]
    ).to_csv(temp_output_file, index=False)

remaining_data = [item for item in data if item.get("id") not in completed_ids]
print(f"Processing {len(remaining_data)} remaining questions...")

batch_size = 8

for i in range(0, len(remaining_data), batch_size):
    items = remaining_data[i : i + batch_size]

    # ── Step 1: Initial generation ────────────────────────────────────────
    prompts = []
    for item in items:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text  = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    all_responses = generate_batch(prompts, sampling_params_initial)

    # ── Step 2: Two paths based on whether boxed answer exists ────────────
    # Path A — no boxed in tail: continue reasoning from where it left off
    # Path B — has boxed in tail: verify / double-check the answer
    needs_continuation = []   # Path A indices
    needs_verification = []   # Path B indices

    for j, resp in enumerate(all_responses):
        if not has_boxed_in_tail(resp, tail_words=150):
            needs_continuation.append(j)
        else:
            needs_verification.append(j)

    print(f"  Batch {i // batch_size + 1}: {len(needs_continuation)} continue, {len(needs_verification)} verify")

    all_step2_responses = [None] * len(items)

    # ── Path A: continuation ──────────────────────────────────────────────
    if needs_continuation:
        cont_prompts = []
        for j in needs_continuation:
            item = items[j]
            system, user = build_prompt(item["question"], item.get("options"))
            prompt_text = tokenizer.apply_chat_template(
                [
                    {"role": "system",    "content": system},
                    {"role": "user",      "content": user},
                    {"role": "assistant", "content": all_responses[j][-2000:]},
                    {"role": "user",      "content": SYSTEM_PROMPT_CONTINUE},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            cont_prompts.append(prompt_text)

        cont_outputs  = generate_batch(cont_prompts,  sampling_params_step2)
        for idx, j in enumerate(needs_continuation):
            all_step2_responses[j] = cont_outputs[idx]

    # ── Path B: verification ──────────────────────────────────────────────
    if needs_verification:
        verif_prompts = []
        for j in needs_verification:
            item = items[j]
            system, user = build_prompt(item["question"], item.get("options"))
            content = all_responses[j]
            if len(content) > 2000:
                # keep first 1000 (reasoning) + last 1000 (boxed answer)
                content = content[:1000] + "\n...\n" + content[-1000:]
            prompt_text = tokenizer.apply_chat_template(
                [
                    {"role": "system",    "content": system},
                    {"role": "user",      "content": user},
                    {"role": "assistant", "content": content},
                    {"role": "user",      "content": SYSTEM_PROMPT_VERIFY},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            verif_prompts.append(prompt_text)

        verif_outputs = generate_batch(verif_prompts, sampling_params_step2)
        for idx, j in enumerate(needs_verification):
            all_step2_responses[j] = verif_outputs[idx]

    # ── Collect & checkpoint ──────────────────────────────────────────────
    rows = []
    for j, item in enumerate(items):
        initial = all_responses[j]
        step2   = all_step2_responses[j]

        # For Path A (continuation): use step2 if it now has boxed, else initial
        # For Path B (verification): step2 always has boxed (model restates or corrects)
        if step2 is not None and has_boxed_in_tail(step2, tail_words=150) and reflection_answer_is_novel(step2, item["question"], item.get("options")):
            final   = step2
            has_box = True
        else:
            # step2 failed novelty check or still no box — fall back to initial
            final   = initial
            has_box = has_boxed_in_tail(initial, tail_words=150)

        rows.append({
            "id":               item.get("id"),
            "question":         item["question"],
            "initial_response": initial,
            "step2_response":   step2,
            "final_response":   final,
            "has_box":          has_box,
        })

    pd.DataFrame(rows).to_csv(temp_output_file, mode="a", header=False, index=False)
    print(f"    -> saved {len(rows)} rows")

print("Done. All batches written to checkpoint.")


Resuming from checkpoint. 736 items already processed.
Processing 207 remaining questions...


Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Batch 1: 0 continue, 8 verify


Adding requests:   0%|          | 0/8 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/7 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Batch 26: 0 continue, 7 verify


Adding requests:   0%|          | 0/7 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/7 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

    -> saved 7 rows
Done. All batches written to checkpoint.


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [17]:
#for submission

import pandas as pd

df = pd.read_csv("data/responses_checkpoint.csv")

submission = df[["id", "final_response"]].rename(columns={"final_response": "response"})
submission.to_csv("data/submission.csv", index=False)
print(f"Saved {len(submission)} rows to data/submission.csv")
print(submission.head(3))

Saved 943 rows to data/submission.csv
   id                                           response
0   0  Okay, let's double-check the problem step by s...
1   1  Okay, let's try to figure out this problem aga...
2   2  Okay, let's go through this problem step by st...


In [ ]:
import pandas as pd

df = pd.read_csv("data/responses_checkpoint.csv")
boxed = (df["has_box"] == True).sum()
print(boxed/943)


In [ ]:
import pandas as pd, re, sys
from tqdm import tqdm

# Load all responses from checkpoint
df = pd.read_csv("data/responses_checkpoint.csv")
print(f"Loaded {len(df)} responses from checkpoint")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", str(text).upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

# Build a lookup from id -> item for gold answers
data_lookup = {item["id"]: item for item in data}

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    item = data_lookup.get(row["id"])
    if item is None:
        continue
    response = row["revised_response"]
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        correct = score_mcq(str(response), str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":      row["id"],
        "is_mcq":  is_mcq,
        "gold":    gold,
        "response": response,
        "correct": correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!